In [1]:
! pip install requests pandas matplotlib numpy

Editar para reusar:

In [2]:
from pathlib import Path

PROJECT_PATH = Path("/Users/daragama/Documents/Master25/Tesis/datos_semisinteticos/")
DATAPATH = PROJECT_PATH  / "data"
print(f"DATAPATH: {DATAPATH}")

DATAPATH: /Users/daragama/Documents/Master25/Tesis/datos_semisinteticos/data


# Revisemos 2024

In [3]:
import pandas as pd
import numpy as np


year = 2024
YEAR_DATAPATH = DATAPATH / "raw" / f"{year}"
raw_deaths_data_path = YEAR_DATAPATH /  "conjunto_de_datos"

year_data_file = list(raw_deaths_data_path.glob("*.csv"))

# open csv in a pandas dataframe
year_data = pd.read_csv(year_data_file[0], low_memory=False) # low_memory=False added to avoid DtypeWarning: Columns (0: maternas) have mixed types.

year_data

,ent_regis,mun_regis,tloc_regis,loc_regis,ent_resid,mun_resid,tloc_resid,loc_resid,ent_ocurr,mun_ocurr,...,complicaro,dia_cert,mes_cert,anio_cert,maternas,ent_ocules,mun_ocules,loc_ocules,razon_m,dis_re_oax
0,1,2,5,1,1,1,15,1,1,5,...,8,23,1,2024,NaN,88,888,8888,0,999
1,1,2,5,59,1,2,1,15,1,1,...,8,17,1,2024,NaN,88,888,8888,0,999
2,1,3,8,1,1,3,5,55,1,1,...,8,12,1,2024,NaN,88,888,8888,0,999
3,1,3,8,1,1,3,2,33,1,3,...,8,17,1,2024,NaN,88,888,8888,0,999
4,1,2,5,1,1,1,15,1,1,1,...,9,14,12,2023,NaN,88,888,8888,0,999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819667,32,10,13,1,99,999,99,9999,32,10,...,8,16,2,2024,NaN,99,999,9999,0,999
819668,32,56,13,1,32,56,13,1,32,56,...,9,4,1,2024,NaN,88,888,8888,0,999
819669,32,56,13,1,14,61,4,1,32,56,...,9,28,11,2024,NaN,88,888,8888,0,999
819670,32,56,13,1,32,55,6,1,32,56,...,9,6,1,2024,NaN,88,888,8888,0,999


Los datos están "códificados, cada columna tiene un catálogo para saber a que corresponde cada valor.

# "Decodificar"

In [4]:
data_dic = YEAR_DATAPATH / "diccionario_de_datos"
dic_file = list(data_dic.glob("*.csv"))[0]
data_dictionary = pd.read_csv(dic_file, low_memory=False, encoding='utf-8') # low_memory=False added to avoid DtypeWarning: Columns (0: maternas) have mixed types.
data_dictionary.rename(columns={'NEMÃ“NICO': 'NEMONICO', 'CATÃLOGO' : 'CATALOGO'}, inplace=True) # no es buena práctica usar acentos en nombres de columnas
data_dictionary

,NOMBRE_CAMPO,LONGITUD,TIPO,NEMONICO,CATALOGO,RANGO_CLAVES
0,Entidad de registro,2,C,ent_regis,entidad_municipio_localidad_2024,01 … 32\t
1,Municipio o demarcaciÃ³n territorial de registro,3,C,mun_regis,entidad_municipio_localidad_2024,001 … 570 ...
2,TamaÃ±o de localidad de registro,2,N,tloc_regis,tamaÃ±o_localidad,"1 … 17, 99 ..."
3,Localidad de registro,4,C,loc_regis,entidad_municipio_localidad_2024,"0001 … 6999, 7777 ..."
4,Entidad de residencia habitual del (la) fallec...,2,C,ent_resid,entidad_municipio_localidad_2024,"01 … 35, 99 ..."
...,...,...,...,...,...,...
69,Entidad de ocurrencia de la lesiÃ³n,2,C,ent_ocules,entidad_municipio_localidad_2024,"01 … 35, 88, 99 ..."
70,Municipio o demarcaciÃ³n territorial de ocurre...,3,C,mun_ocules,entidad_municipio_localidad_2024,"001 … 570, 888, 999 ..."
71,Localidad de ocurrencia de la lesiÃ³n,4,C,loc_ocules,entidad_municipio_localidad_2024,"0001 … 6999, 7777, 8888, 9999 ..."
72,Defunciones para calcular la razÃ³n de la mort...,1,N,razon_m,razon_materna,"1, Nulo"


In [5]:
# el archivo está mal códigificado, por lo que se debe reparar el mojibake
def reparar_mojibake(texto):
    if isinstance(texto, str):
        try:
            return texto.encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return texto
    return texto

In [6]:
# reparar data_dictionary
data_dictionary = data_dictionary.map(reparar_mojibake)
data_dictionary.head()

,NOMBRE_CAMPO,LONGITUD,TIPO,NEMONICO,CATALOGO,RANGO_CLAVES
0,Entidad de registro,2,C,ent_regis,entidad_municipio_localidad_2024,01 … 32\t
1,Municipio o demarcación territorial de registro,3,C,mun_regis,entidad_municipio_localidad_2024,001 … 570 ...
2,Tamaño de localidad de registro,2,N,tloc_regis,tamaño_localidad,"1 … 17, 99 ..."
3,Localidad de registro,4,C,loc_regis,entidad_municipio_localidad_2024,"0001 … 6999, 7777 ..."
4,Entidad de residencia habitual del (la) fallec...,2,C,ent_resid,entidad_municipio_localidad_2024,"01 … 35, 99 ..."


## ENTIDAD

In [7]:
item_dic = data_dictionary.iloc[0]
nemonico = item_dic['NEMONICO']
catalogo = item_dic['CATALOGO'] + ".csv"
catalogo_path = YEAR_DATAPATH / "catalogos" / catalogo
df_catalogo = pd.read_csv(catalogo_path, low_memory=False)

In [8]:
# Seleccionar solo las entidades y columnas de interes
mapeo_ent = df_catalogo.loc[df_catalogo['cve_mun']==0, ['cve_ent','nom_loc']]
# cambiar nombre columnas
mapeo_ent.columns = ['ent_code', 'ent_nom']
display(mapeo_ent)

,ent_code,ent_nom
0,1,Aguascalientes
157,2,Baja California
296,3,Baja California Sur
362,4,Campeche
608,5,Coahuila de Zaragoza
840,6,Colima
951,7,Chiapas
3248,8,Chihuahua
3823,9,Ciudad de México
3883,10,Durango


In [ ]:
columnas_de_ent = ["ent_regis", "ent_resid", "ent_ocurr", "ent_nac"]
year_data_fixed = year_data.copy()
mapeo_ent = dict(zip(mapeo_ent["ent_code"], mapeo_ent["ent_nom"]))

for col in columnas_de_ent:
    year_data_fixed[col] = year_data_fixed[col].map(mapeo_ent)

## MUNICIPIO


In [10]:
# Seleccionar solo las entidades y columnas de interes
mapeo_mun = df_catalogo.loc[df_catalogo['cve_loc']==0, ['cve_ent', 'cve_mun','nom_loc']]
# cambiar nombre columnas
mapeo_mun.columns = ['ent_code', 'mun_code', 'mun_nom']
mapeo_mun

,ent_code,mun_code,mun_nom
0,1,0,Aguascalientes
1,1,1,Aguascalientes
28,1,2,Asientos
54,1,3,Calvillo
72,1,4,Cosío
...,...,...,...
28489,33,999,Municipio no especificado
28491,88,0,Entidad no aplica para A00 - R99 Y V90 - Y89
28492,88,888,Municipio no aplica para A00 - R99 Y V90 - Y89
28494,99,0,Entidad no especificada


In [11]:
mapeo_mun = dict(zip(mapeo_mun["mun_code"], mapeo_mun["mun_nom"]))

columnas_de_mun = ["mun_regis", "mun_resid", "mun_ocurr"]

for col in columnas_de_mun:
    year_data_fixed[col] = year_data_fixed[col].map(mapeo_mun)

## LOCALIDAD REGISTRADA

In [12]:

item_dic = data_dictionary.iloc[3]
nemonico = item_dic['NEMONICO']
catalogo = item_dic['CATALOGO'] + ".csv"
catalogo_path = YEAR_DATAPATH / "catalogos" / catalogo
df_catalogo = pd.read_csv(catalogo_path, low_memory=False)

catalogue_col_code = "cve_loc"
catalogue_col_name = "nom_loc"

mapeo = df_catalogo.set_index(catalogue_col_code)[catalogue_col_name].to_dict()

year_data_fixed[nemonico] = year_data[nemonico].map(mapeo)


print(f"Valores únicos en RAW data : {len(year_data[nemonico].unique())}")
print(f"Valores únicos en FIXED data : {len(year_data_fixed[nemonico].unique())}")


Valores únicos en RAW data : 310
Valores únicos en FIXED data : 305


> Hay algunas localidades que tienen doble código, por ejemplo 07004,07,004,0000,Altamirano y 070040001,07,004,0001,Altamirano

## SEXO

> Igual  que https://github.com/mar-esther23/CursoPython_DefuncionesCOVID19enMexico/blob/main/EDR4_Limpieza.ipynb solo cambié .replace por .map

In [13]:
# Definir el diccionario de mapeo para la columna sexo
mapeo_sexo = { 1:'Hombre', 2:'Mujer', 9:'No especificado' }
year_data_fixed["sexo"] = year_data["sexo"].map(mapeo_sexo)

year_data_fixed['sexo'].value_counts(dropna=False)

sexo
Hombre             458266
Mujer              360814
No especificado       592
Name: count, dtype: int64

In [14]:
year_data_fixed['sexo'] = year_data_fixed['sexo'].astype('category')
year_data_fixed['sexo']

0                  Hombre
1                  Hombre
2                  Hombre
3                  Hombre
4                   Mujer
               ...       
819667    No especificado
819668              Mujer
819669              Mujer
819670              Mujer
819671              Mujer
Name: sexo, Length: 819672, dtype: category
Categories (3, str): ['Hombre', 'Mujer', 'No especificado']

## EDAD

In [15]:
# Definimos la función especial
def modify_age_inegi(number):
    if number == 4998:
        return np.nan
    elif number < 4000:
        return 0
    else:
        return number - 4000

# Aplicar la función a la columna 'EDAD' y crear una nueva columna 'EDAD_ANOS'
year_data_fixed['edad'] = year_data_fixed['edad'].apply(modify_age_inegi)

# Mostrar los valores únicos de la nueva columna para verificar el cambio
year_data_fixed['edad'].value_counts(dropna=False).sort_index()

edad
0.0      17421
1.0       1663
2.0        905
3.0        616
4.0        595
         ...  
117.0        3
118.0        2
119.0        1
120.0        4
NaN       4522
Name: count, Length: 122, dtype: int64

## EDAD AGRUPADA
Será útil para generar nombres

In [16]:
# igual que el curso
from pandas.api.types import CategoricalDtype
# Construir el catálogo
mapeo_edad_agru = {
                    1:'Menores de un año', 2:'De un año', 3:'De 2', 4:'De 3', 5:'De 4', 6:'De 5 a 9',
                    7:'De 10 a 14', 8:'De 15 a 19', 9:'De 20 a 24', 10:'De 25 a 29', 11:'De 30 a 34',
                    12:'De 35 a 39', 13:'De 40 a 44', 14:'De 45 a 49', 15:'De 50 a 54', 16:'De 55 a 59',
                    17:'De 60 a 64', 18:'De 65 a 69', 19:'De 70 a 74', 20:'De 75 a 79', 21:'De 80 a 84',
                    22:'De 85 a 89', 23:'De 90 a 94', 24:'De 95 a 99', 25:'De 100 a 104', 26:'De 105 a 109',
                    27:'De 110 a 114', 28:'De 115 a 119', 29:'De 120 y más', 30:'No especificada'
                  }
# Aplicar el reemplazo en la columna 'edad_agru'
year_data_fixed['edad_agru'] = year_data['edad_agru'].replace(mapeo_edad_agru)
# Generar la categoría ordenada, usamos los valores del diccionario
cat_edad_agru = CategoricalDtype(categories=mapeo_edad_agru.values(), ordered=True)
# Convertir a la columna a la categoría ordenada
year_data_fixed['edad_agru'] = year_data_fixed['edad_agru'].astype(cat_edad_agru)
# Mostrar el conteo de valores únicos para verificar el cambio
year_data_fixed['edad_agru'].value_counts().sort_index()

edad_agru
Menores de un año    17421
De un año             1663
De 2                   905
De 3                   616
De 4                   595
De 5 a 9              2425
De 10 a 14            3329
De 15 a 19            9701
De 20 a 24           14723
De 25 a 29           17381
De 30 a 34           20769
De 35 a 39           22259
De 40 a 44           26970
De 45 a 49           35118
De 50 a 54           45233
De 55 a 59           54749
De 60 a 64           66875
De 65 a 69           76118
De 70 a 74           79988
De 75 a 79           84340
De 80 a 84           83905
De 85 a 89           75030
De 90 a 94           50086
De 95 a 99           19846
De 100 a 104          4588
De 105 a 109           449
De 110 a 114            53
De 115 a 119            11
De 120 y más             4
No especificada       4522
Name: count, dtype: int64

## FECHAS

In [ ]:
fechas = {
    "fecha_ocurr": ["dia_ocurr", "mes_ocurr", "anio_ocur"],
    "fecha_regis": ["dia_regis", "mes_regis", "anio_regis"],
    "fecha_nacim": ["dia_nacim", "mes_nacim", "anio_nacim"],
    "fecha_cert": ["dia_cert", "mes_cert", "anio_cert"]
}

for fecha_col, date_parts in fechas.items():
    # Reemplazar los valores 99 y 9999 por NaN en las columnas de fecha
    year_data_fixed[date_parts[0]] = year_data_fixed[date_parts[0]].replace(99, np.nan)
    year_data_fixed[date_parts[1]] = year_data_fixed[date_parts[1]].replace(99, np.nan)
    year_data_fixed[date_parts[2]] = year_data_fixed[date_parts[2]].replace(9999, np.nan)

    # Crear un DataFrame temporal con las partes de la fecha
    data_time = year_data_fixed[date_parts].rename(columns={date_parts[0]: 'day', date_parts[1]: 'month', date_parts[2]: 'year'})
    
    # Generar la columna datetime
    year_data_fixed[fecha_col] = pd.to_datetime(data_time, errors='coerce')  # errors='coerce' para manejar fechas inválidas

    # Mostrar las últimas filas para verificar
    display(year_data_fixed[[*date_parts, fecha_col]].tail())

    # Eliminar las columnas originales de día, mes y año
    year_data_fixed.drop(columns=date_parts, inplace=True)

,dia_ocurr,mes_ocurr,anio_ocur,fecha_ocurr
819667,16.0,2.0,2024.0,2024-02-16
819668,4.0,1.0,2024.0,2024-01-04
819669,28.0,11.0,2024.0,2024-11-28
819670,6.0,1.0,2024.0,2024-01-06
819671,29.0,12.0,2024.0,2024-12-29


,dia_regis,mes_regis,anio_regis,fecha_regis
819667,16.0,2,2024,2024-02-16
819668,4.0,1,2024,2024-01-04
819669,28.0,11,2024,2024-11-28
819670,6.0,1,2024,2024-01-06
819671,29.0,12,2024,2024-12-29


,dia_nacim,mes_nacim,anio_nacim,fecha_nacim
819667,NaN,NaN,NaN,NaT
819668,21.0,3.0,1990.0,1990-03-21
819669,14.0,2.0,2008.0,2008-02-14
819670,1.0,9.0,1949.0,1949-09-01
819671,25.0,5.0,1988.0,1988-05-25


In [18]:
year_data_fixed[['fecha_ocurr', 'fecha_regis', 'fecha_nacim']].info()


<class 'pandas.DataFrame'>
RangeIndex: 819672 entries, 0 to 819671
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   fecha_ocurr  818965 non-null  datetime64[us]
 1   fecha_regis  770189 non-null  datetime64[us]
 2   fecha_nacim  809625 non-null  datetime64[us]
dtypes: datetime64[us](3)
memory usage: 18.8 MB


In [19]:
enr = DATAPATH / "raw" / "enr2024" / "conjunto_de_datos" / "conjunto_de_datos_enr2024.csv"
df_enr = pd.read_csv(enr, low_memory=False)
df_enr

,ent_regis,mun_regis,loc_regis,tloc_regis,ent_resid,mun_resid,loc_resid,tloc_resid,ent_ocurr,mun_ocurr,...,act_pad,pos_mad,pos_pad,sitlab_mad,sitlab_pad,fue_prese,hora_nac,minuto_nac,comparecio,dis_re_oax
0,1,3,55,5,1,3,1,8,1,1,...,1,9,3,2,1,1,18,29,3,999
1,1,7,1,9,1,7,1,9,1,5,...,1,9,1,3,1,1,1,35,3,999
2,1,1,1,15,1,3,1,8,1,1,...,9,9,9,3,9,1,20,0,2,999
3,1,1,1,15,1,1,1,15,1,1,...,1,9,1,3,1,1,12,0,3,999
4,1,1,1,15,1,1,1,15,1,1,...,1,4,4,1,1,1,21,45,3,999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1672222,32,22,1,7,32,22,1,7,32,10,...,1,2,2,1,1,1,11,23,3,999
1672223,32,56,1,13,32,56,9999,99,32,56,...,1,9,2,3,1,1,16,35,3,999
1672224,32,20,1,10,32,20,1,10,32,56,...,1,2,2,1,1,1,2,31,3,999
1672225,32,17,1,13,32,17,9999,99,33,999,...,9,9,9,9,9,1,8,2,2,999
